# Avaliação final no teste oficial — RUL FD001

A busca de hiperparâmetros será encerrada nesta etapa. Como o Extra Trees v1 apresentou o menor erro médio na validação por motor, essa configuração será congelada e o modelo será treinado novamente usando todos os motores disponíveis no treino. O Extra Trees combina muitas árvores de decisão e introduz aleatoriedade na construção delas, o que ajuda a reduzir a dependência de uma única regra de decisão e torna a previsão mais estável (Geurts et al., 2006).

A avaliação será feita no conjunto oficial de teste. Para cada motor, será utilizada a última observação disponível e ela será comparada com o RUL fornecido no arquivo oficial do conjunto de dados. Esse procedimento representa a situação em que o sistema precisa estimar quanto tempo ainda resta quando o histórico observado é interrompido (Saxena et al., 2008).

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.io as pio
from multilang import LangMap

pio.renderers.default = 'vscode'
lang = LangMap(source='pt', target='pt')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split
from cmapss_modeling import add_rul_label, candidate_rul_features, nonconstant_features
from cmapss_optuna import build_pipeline_from_params, optimize_model
from cmapss_evaluation import (
    evaluate_official_test,
    load_official_rul,
    summarize_signed_error,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'

## 1. Preparação e congelamento da configuração

A mesma preparação usada na validação será repetida. A configuração vencedora será recuperada do estudo persistido no SQLite; portanto, essa célula não abre uma nova busca nem modifica a decisão tomada na etapa anterior.

In [2]:
train = add_rul_label(load_split(DATA_DIR, 'train', 'FD001'))
test = load_split(DATA_DIR, 'test', 'FD001')
features = nonconstant_features(
    train,
    candidate_rul_features(include_settings=True),
)
study = optimize_model(
    train,
    features=features,
    model_name='Extra Trees',
    n_trials=100,
    n_splits=5,
    random_state=42,
    storage_dir=PROJECT_ROOT / 'outputs' / 'optuna',
    search_version='v1',
)
model = build_pipeline_from_params(
    model_name='Extra Trees',
    params=study.best_params,
    n_features=len(features),
    random_state=42,
    search_version='v1',
)
pd.DataFrame([{
    'modelo': 'Extra Trees',
    'mae_validacao_motor': study.best_value,
    'melhor_trial': study.best_trial.number,
    'hiperparametros': ' | '.join(f'{k}={v}' for k, v in study.best_params.items()),
}])

[I 2026-09-28 14:22:12,537] Using an existing study with `study_name='fd001_extra_trees_v1'` instead of creating a new one.


,modelo,mae_validacao_motor,melhor_trial,hiperparametros
0,Extra Trees,25.014615,77,feature_selection=passthrough | n_estimators=4...


## 2. Avaliação no conjunto oficial de teste

O MAE informa, em média, quantos ciclos o modelo erra. O RMSE dá peso maior aos erros muito grandes. O erro médio mostra a direção do desvio: valores positivos indicam que o RUL foi superestimado, enquanto valores negativos indicam que foi subestimado.

In [3]:
official_rul = load_official_rul(DATA_DIR, 'FD001')
test_results, test_metrics = evaluate_official_test(
    model=model,
    train_df=train,
    test_df=test,
    features=features,
    official_rul=official_rul,
)
pd.DataFrame([test_metrics]).round(2)

,mae,rmse,erro_medio,mae_mediano_motor
0,18.66,24.75,10.28,12.58


### Insights

Os valores acima representam a primeira medição fora do processo de escolha do modelo. A comparação deve ser feita com cuidado: o resultado do teste não deve ser usado para voltar e escolher outra configuração, pois isso faria o teste participar indiretamente do treinamento. Caso o erro seja maior que o observado na validação, será sinal de que existe diferença entre os motores usados para orientar a busca e os motores reservados para a avaliação final.

## 3. Erro por motor

A figura mostra a relação entre o RUL real e o RUL previsto. Quanto mais próximos os pontos estiverem da linha diagonal, mais próxima estará a estimativa do valor de referência. A identificação do motor e o último ciclo observado permanecem disponíveis ao passar o cursor sobre cada ponto.

In [4]:
axis_limit = max(test_results['rul_real'].max(), test_results['rul_predito'].max()) * 1.05
fig = px.scatter(
    test_results,
    x='rul_real',
    y='rul_predito',
    hover_data=['unit', 'last_cycle', 'erro'],
    labels={
        'rul_real': 'RUL real (ciclos)',
        'rul_predito': 'RUL previsto (ciclos)',
        'unit': 'Motor',
        'last_cycle': 'Último ciclo observado',
        'erro': 'Erro (ciclos)',
    },
    title=lang({'title': 'RUL real versus RUL previsto — teste FD001'})['title'],
    height=600,
)
fig.add_shape(
    type='line',
    x0=0, y0=0, x1=axis_limit, y1=axis_limit,
    line={'dash': 'dash', 'color': 'gray'},
)
fig.update_xaxes(range=[0, axis_limit])
fig.update_yaxes(range=[0, axis_limit])
fig.show()

### Insights

A dispersão dos pontos será utilizada para identificar se o erro está concentrado em motores específicos ou em determinada faixa de vida útil. Essa leitura será importante antes de transformar a previsão em alerta operacional, porque um erro médio aceitável pode esconder falhas grandes justamente nos motores próximos da falha.

## 4. Verificação da direção do erro

O erro assinado será analisado separadamente do MAE. Valores positivos representam RUL previsto acima do RUL real e, portanto, superestimação do tempo restante. Valores negativos representam subestimação. Essa distinção é importante porque os dois erros podem ter o mesmo tamanho numérico, mas consequências operacionais diferentes.

In [5]:
signed_error_summary = summarize_signed_error(test_results)
signed_error_summary.round(2)

,erro_medio,erro_mediano,percentual_superestimacao,percentual_subestimacao,percentual_exato,erro_p10,erro_p90,maior_superestimacao,maior_subestimacao,motores
0,10.28,6.58,69.0,31.0,0.0,-14.21,37.4,68.08,-41.42,100


### Insights

A média positiva observada anteriormente indica superestimação agregada, mas a mediana e os percentuais mostram se essa tendência está espalhada entre os motores ou concentrada em poucos casos. Os percentis e os extremos serão usados para distinguir um viés sistemático de alguns erros muito grandes.

In [6]:
error_direction = test_results.assign(
    direcao_erro=test_results['erro'].map(
        lambda value: 'Superestimação' if value > 0 else 'Subestimação' if value < 0 else 'Exato'
    )
)
fig = px.histogram(
    error_direction,
    x='erro',
    color='direcao_erro',
    nbins=20,
    labels={'erro': 'Erro assinado: previsto − real', 'direcao_erro': 'Direção'},
    title=lang({'title': 'Distribuição do erro assinado por motor'})['title'],
    height=550,
)
fig.add_vline(x=0, line_dash='dash', line_color='black')
fig.show()

### Insights

A distribuição acima permite verificar se o modelo está deslocado para o lado positivo. Caso a maior parte dos motores esteja à direita de zero, haverá uma tendência de atraso potencial na manutenção. Caso poucos motores concentrem os maiores valores positivos, deverá ser investigado se eles representam um subgrupo específico de trajetória.

In [7]:
top_overestimation = test_results.nlargest(10, 'erro')[
    ['unit', 'last_cycle', 'rul_real', 'rul_predito', 'erro']
].reset_index(drop=True)
top_underestimation = test_results.nsmallest(10, 'erro')[
    ['unit', 'last_cycle', 'rul_real', 'rul_predito', 'erro']
].reset_index(drop=True)
top_overestimation.round(2)

,unit,last_cycle,rul_real,rul_predito,erro
0,15,76,83.0,151.08,68.08
1,1,31,112.0,176.75,64.75
2,67,71,77.0,136.04,59.04
3,22,39,111.0,165.08,54.08
4,78,72,107.0,157.73,50.73
5,2,49,98.0,146.54,48.54
6,50,74,79.0,127.16,48.16
7,41,123,18.0,63.80,45.80
8,85,34,118.0,160.03,42.03
9,27,140,66.0,104.39,38.39


Os dez maiores casos de superestimação serão inspecionados antes dos dez maiores casos de subestimação. Se a superestimação estiver concentrada em poucos motores, uma análise por trajetória poderá revelar um padrão de comportamento que não aparece no MAE global.

In [8]:
top_underestimation.round(2)

,unit,last_cycle,rul_real,rul_predito,erro
0,45,152,114.0,72.58,-41.42
1,89,177,136.0,97.70,-38.30
2,12,217,124.0,88.74,-35.26
3,74,137,126.0,91.07,-34.93
4,30,143,115.0,80.64,-34.36
5,93,244,85.0,51.73,-33.27
6,57,160,103.0,76.60,-26.40
7,8,166,95.0,72.74,-22.26
8,51,144,114.0,91.90,-22.10
9,55,113,137.0,119.65,-17.35


## Referências

GEURTS, P.; ERNST, D.; WEHENKEL, L. Extremely randomized trees. Machine Learning, v. 63, p. 3–42, 2006.

SAXENA, A. et al. Damage propagation modeling for aircraft engine run-to-failure simulation. In: Proceedings of the International Conference on Prognostics and Health Management, 2008.